<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/03-rag/00-what-is-rag.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 什么是 RAG

**目标：** 理解什么是 Retrieval-Augmented Generation（检索增强生成），它解决了哪些问题，以及 retrieve → augment → generate 这个循环。然后在本节其余部分把每一步都正经搭起来之前，先用约 15 行把整条链路跑起来。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 环境准备

每个 notebook 都是自包含的，接下来的两个单元格会把它从零搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的环境准备 helper，并拉入 `groq` 客户端——这个 notebook 唯一需要的依赖。
2. **载入你的 API key。** 免费 key 在 [console.groq.com](https://console.groq.com/) 领（无需信用卡）。在 Colab 里通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称必须严格写成 `GROQ_API_KEY`，粘贴值，并把 **Notebook access** 打开。本地运行的话，则把 `GROQ_API_KEY` 设成环境变量。

（完整走查和选模型的指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 载入 GROQ_API_KEY（Colab Secrets 或本地环境变量）并返回一个可直接
# 使用的 Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用
# 报 404，就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## RAG 是什么

一个语言模型只知道它训练数据里有的东西。问它你公司内部的 runbook、上周的故障，或者任何私有文档，它要么拒答，要么更糟——自信地编一个错答案。

**Retrieval-Augmented Generation（RAG）** 用一个动作解决这件事：在让模型回答*之前*，你从一个外部来源 **retrieve（检索）** 出相关文本，并把它 **粘贴进 prompt** 当作上下文。接着模型就从你交给它的内容里作答，而不是只靠记忆。Retrieve → 增强 prompt → generate。

2020 年那篇原始论文（Lewis 等）把这件事描述成组合两类记忆：

- **参数化记忆 ——** 训练时烘焙进模型权重里的东西（固定、通用、无法引用）。
- **非参数化记忆 ——** 一个外部存储，你可以随时更新（你的文档、数据库、API），模型在作答时读取它。

RAG 只是把两者连起来的管道。整个思路就这么多；本节其余内容讲的都是*怎样把每一步做好*。

## 为什么要 RAG：它解决的四个问题

一次朴素的 LLM 调用有四种失效模式，RAG 直接对付它们：

| 朴素 LLM 的问题 | RAG 如何解决 |
|---|---|
| **知识截止：** 训练数据是冻结的；它不知道近期发生的事 | 从你保持更新的来源里检索，无需重训 |
| **幻觉：** 即便它不知道，也会自信作答 | 让答案落地在可被引用的检索证据上 |
| **没有私有/领域数据：** 它从没见过你的内部文档 | 在查询时把这些文档作为上下文提供出去 |
| **没有引用：** 你没法判断答案从哪来 | 答案指回被检索到的段落 |

还有第五个，很实际：**成本**。通过*微调*或重训来注入知识又贵又慢；RAG 注入知识只要……把它放进 prompt。这就是为什么当一个模型需要知道它没被训练过的东西时，RAG 通常是你第一个想到的手段。（微调 vs RAG vs prompt 的完整决策，以及为什么微调是处理知识的*错*工具，见第 06 节 [让模型适配](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/06-adaptation/01-fine-tune-vs-rag-vs-prompt.zh.ipynb)。）

**什么时候*不*该用 RAG：** 如果任务不需要外部事实（总结这封邮件、改写这段话），检索只会白白增加延迟和成本。只有当答案依赖于模型之外的知识时，RAG 才配得上它的位置。

## RAG 一定得用 embedding 吗？不是。

> **⭐ 关键收获 ——** 最常见的那个误解：**RAG 不等同于 embedding 或向量数据库。** RAG 的意思是*先检索、再生成*，而*怎么*检索是一个你可以更换的实现选择。

- **Embedding + 向量搜索 ——** 流行的默认做法（notebook 01）。当用户的用词和文档不一致时，它擅长匹配*含义*。
- **关键词搜索（BM25） ——** 完全不含神经网络（notebook 02）。对于精确的标识符、错误码、名称，它常常*胜过* embedding。
- **混合检索 ——** 两者同时用，再融合（notebook 02）。大多数生产系统实际在用的就是它。
- **一条 SQL 查询、一次 API 调用、一次普通的文档查找 ——** 只要它取回相关上下文放进 prompt，它就是 RAG。

下面的 demo 正好证明这一点。它检索时*只用字符串匹配*，任何地方都没有 embedding，但它仍然是 RAG。我们在 notebook 01 里转向 embedding，是因为关键词匹配会漏掉同义改写，而不是因为 RAG 要求用 embedding。

## 整条流水线，以及本节在哪里把它搭起来

每个 RAG 系统都有四个阶段。数据按下表顺序流动，但本节是按*发现*顺序去讲的：先把检索跑通，等你有了判断力再回头处理分块决策。

| 阶段 | 发生什么 | 在第几篇搭建 |
|---|---|---|
| **1. 索引** | 把源文档切成 chunk、embed、存起来 | `01-embeddings-retrieval`（embed + 存储）、`03-chunking`（切分决策，最后回头再看） |
| **2. 检索** | 给定一个查询，找出最相关的 chunk | `01-embeddings-retrieval`（向量）、`02-hybrid-and-reranking`（关键词、混合、重排序） |
| **3. 增强** | 把检索到的 chunk 粘进 prompt 当作上下文 | 见下面的 demo；全节都在用 |
| **4. 生成** | 模型基于该上下文作答，并带引用 | 见下面的 demo；每一篇里都会打磨 |

为什么分块（`03`）放在检索*之后*讲，哪怕它执行起来更靠前？因为只有当你亲眼看过检索在它身上成功和失败之后，分块选择才说得通。chunk 大小是与 embedding 模型、检索精度、prompt 预算之间的权衡，而这些东西你会在 `01` 和 `02` 里遇到。然后 `04-why-rag-fails` 讲的是当答案很糟时如何诊断整条链路——剧透一下，这几乎总是*检索*的问题，而不是生成的问题。

先让我们用尽可能小的形式，一次看完这四个阶段。

## 约 15 行的 RAG

一个三篇文档的「知识库」、一个只按共享词打分的玩具检索器（没有 embedding！），以及一次*只*依据检索结果作答的生成调用。这就是一个完整的 RAG 系统，本节其余内容都在把每个阶段做得更好。

In [ ]:
# 1. 索引：一个微小的知识库。（在真实系统里，这些是你文档切出来的 chunk。）
DOCS = [
    "The Acme return policy allows refunds within 30 days of purchase with a receipt.",
    "Acme support hours are Monday to Friday, 9am to 6pm Pacific time.",
    "Acme's enterprise plan includes a dedicated account manager and a 99.9% uptime SLA.",
]

# 2. 检索：按每篇文档包含多少个查询词来打分。没有 embedding ——
#    刻意用最能跑通的最笨检索器，以此证明 检索 != embedding。
def retrieve(query, k=1):
    q_words = set(query.lower().split())
    scored = [(len(q_words & set(d.lower().split())), d) for d in DOCS]
    scored.sort(reverse=True)                       # 重叠最多的排最前
    return [d for score, d in scored[:k] if score]  # 丢掉零重叠的文档

# 3. 增强 + 4. 生成：把检索到的文档粘进 prompt，只依据它们作答。
def rag_answer(question, k=1):
    context = "\n".join(retrieve(question, k)) or "(no relevant documents found)"
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context does not contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=150,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content

question = "How long do I have to return something to Acme?"
print("RETRIEVED:", retrieve(question))
print("\nANSWER:", rag_answer(question))

跑一下。检索器把退货政策那篇文档捞了出来（它和问题的共享词最多），这段文本进了 prompt，模型*依据它*作答：「30 天」——落地在一个它从未被训练过的事实上。

现在试试那个引出整节后续内容的失效模式：

In [ ]:
# 一个答案就在知识库里、但用词和文档不同的查询。
# 看关键词检索器怎么漏掉它。
q2 = "When can I reach a human at Acme for help?"   # 文档写的是 "support hours"，不是 "reach a human"
print("RETRIEVED:", retrieve(q2))
print("\nANSWER:", rag_answer(q2))

取决于词重叠的程度，关键词检索器可能抓错文档或什么都抓不到，即使*客服时间*那篇文档明明能回答这个问题——因为查询和文档几乎没有任何字面相同的词（「reach a human」对「support hours」）。**模型的上限，就是检索交给它的东西。** 就这一个失败，正是接下来几篇 notebook 存在的理由：

- **Notebook 01** 用 *embedding* 取代词重叠，让「reach a human」和「support hours」按含义匹配上。
- **Notebook 02** 把关键词搜索作为*互补*手段加回来（对付精确标识符它仍是最好的工具），并把两者融合。
- **Notebook 03** 回到分块决策。现在你见过检索了，可以推理切分是如何影响它的。
- **Notebook 04** 是一门诊断这类 bug 的手艺，用在真实系统里。

而且这些你都不会靠感觉去调：你在 **第 02 节**里已经养成了评测习惯（「先度量，再调优」）。往后的每一个检索选择（embedding 模型、混合权重、chunk 大小）都由一个数字来判定，而不是靠直觉，第 04 节会把这件事对整个 RAG 系统做得很严谨。

现在你已经看完整条循环了。本节其余部分是对每个阶段的深入。

## 练习

1. **加一篇文档和一个针对它的问题。** 往 `DOCS` 里追加第四条事实（比如发货时间），然后问一个它能回答的问题。确认检索器把它翻了出来、模型也用上了它。
2. **故意弄坏它。** 问一个答案*完全不*在 `DOCS` 里的问题（例如「Acme 的股价是多少？」）。确认模型说自己不知道，而不是编一个答案；这就是 RAG 为你买来的 grounding。然后把 prompt 里「只用上下文」那条指令删掉，看行为如何变化。
3. **调高 `k`。** 调用 `rag_answer(question, k=3)`，让三篇文档全部进入上下文。答案质量变了吗？这就是 notebook 01 和 02 所形式化的 精度 vs 召回 张力：更多上下文意味着更有可能包含答案，但也意味着更多无关文本稀释了 prompt。
4. **说出检索方式的名字。** 各用一句话，描述你会怎样把玩具 `retrieve()` 换成（a）一次 embedding 搜索、（b）对客户数据库的一条 SQL 查询、（c）一个实时的天气 API。三者都仍然是 RAG，所以说说为什么。